In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from empiricaldist import Pmf, Cdf
from scipy.stats import norm

In [2]:
# function to normalize the posterior

def normalize(joint):
    """Normalize the joint distribution."""
    prob_data = joint.to_numpy().sum()
    joint /= prob_data
    return prob_data

In [3]:
df = pd.read_csv('drp_scores.csv')

In [4]:
grouped = df.groupby('Treatment')
responses = {}

for name, group in grouped:
    responses[name] = group['Response']

In [5]:
def make_cdf_map(df, colname):
    """Make a map from treatment to CDF."""
    grouped = df.groupby('Treatment')
    cdf_map = {}
    for name, group in grouped:
        cdf_map[name] = Cdf.from_seq(group[colname], name=name)
    return cdf_map

In [6]:
make_cdf_map(df, 'Response')

{'Control': Response
 10    0.043478
 17    0.086957
 19    0.130435
 20    0.173913
 26    0.217391
 28    0.260870
 33    0.304348
 37    0.391304
 41    0.434783
 42    0.565217
 43    0.608696
 46    0.652174
 48    0.695652
 53    0.739130
 54    0.782609
 55    0.869565
 60    0.913043
 62    0.956522
 85    1.000000
 Name: Control, dtype: float64,
 'Treated': Response
 24    0.047619
 33    0.095238
 43    0.238095
 44    0.285714
 46    0.333333
 49    0.428571
 52    0.476190
 53    0.523810
 54    0.571429
 56    0.619048
 57    0.714286
 58    0.761905
 59    0.809524
 61    0.857143
 62    0.904762
 67    0.952381
 71    1.000000
 Name: Treated, dtype: float64}

In [7]:
def make_uniform(qs, name=None, **options):
    """Make a uniform PMF."""
    pmf = Pmf(1.0, qs, **options)
    pmf.normalize()
    if name:
        pmf.index.name = name
    return pmf

In [8]:
qs = np.linspace(20, 80, num=101)
prior_mu = make_uniform(qs, name='mu')

In [9]:
qs = np.linspace(5, 30, num=101)
prior_sigma = make_uniform(qs, name='std')

In [ ]:
def make_joint(pmf1, pmf2):
    """Make a joint distribution from two PMFs."""
    X, Y = np.meshgrid(pmf1, pmf2)
    return pd.DataFrame(X * Y, columns=pmf1.qs, index=pmf2.qs)

In [11]:
prior = make_joint(prior_mu, prior_sigma)

In [12]:
data = responses['Control']
data.shape

(23,)

In [13]:
mu_mesh, sigma_mesh, data_mesh = np.meshgrid(prior.columns, prior.index, data)

In [14]:
mu_mesh.shape

(101, 101, 23)

In [15]:
densities = norm(mu_mesh, sigma_mesh).pdf(data_mesh)
densities.shape

(101, 101, 23)

In [16]:
likelihood = densities.prod(axis=2)
likelihood.shape

(101, 101)